In [3]:
import glob
import pandas as pd  #  type: ignore

files = sorted(glob.glob("../data/*.csv.zip"))

dfs = [pd.read_csv(f, compression="zip") for f in files]

flights = pd.concat(dfs, ignore_index=True)

flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21607106 entries, 0 to 21607105
Data columns (total 23 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY_OF_MONTH         int64  
 3   DAY_OF_WEEK          int64  
 4   FL_DATE              object 
 5   ORIGIN               object 
 6   DEST                 object 
 7   CRS_DEP_TIME         int64  
 8   DEP_TIME             float64
 9   DEP_DELAY            float64
 10  DEP_DEL15            float64
 11  ARR_DELAY            float64
 12  ARR_DEL15            float64
 13  CANCELLED            float64
 14  CANCELLATION_CODE    object 
 15  DIVERTED             float64
 16  FLIGHTS              float64
 17  DISTANCE             float64
 18  CARRIER_DELAY        float64
 19  WEATHER_DELAY        float64
 20  NAS_DELAY            float64
 21  SECURITY_DELAY       float64
 22  LATE_AIRCRAFT_DELAY  float64
dtypes: float64(14), int64(5), obje

In [ ]:
# Estadísticos de las columnas numéricas
flights.describe()

In [ ]:
# Cómputo del total de registros de la tabla
numTotalFlights = len(flights)
numTotalFlights

In [ ]:
# Filtrado de registros para vuelos retrasados
# Un vuelo con un retraso mayor a 15 minutos se considera retrasado
# (columna DepDelay).
delayedFlights = flights[flights["DepDelay"] > 15][["UniqueCarrier", "DepDelay"]]
delayedFlights.head(5)

In [ ]:
# Cálculo del porcentaje de vuelos retrasados
numDelayedFlights = len(delayedFlights)
print(
    "Porcentaje de vuelos retrasados: "
    + str(round(numDelayedFlights / numTotalFlights * 100, 2))
    + "%"
)

In [ ]:
# Copia de una tabla y copia de columnas
flightsWithDelays = flights[
    [
        "Year",
        "Month",
        "DayofMonth",
        "UniqueCarrier",
        "FlightNum",
        "DepDelay",
    ]
].copy()

flightsWithDelays["IsDelayed"] = flightsWithDelays["DepDelay"].copy()

In [ ]:
# Conteo de registros nulos en una columna
flightsWithDelays.IsDelayed.isna().sum()

In [ ]:
# Aplicación de una función a una columna
flightsWithDelays["IsDelayed"] = flightsWithDelays["IsDelayed"].map(
    lambda x: 0 if pd.isna(x) else x
)
flightsWithDelays["IsDelayed"] = flightsWithDelays["IsDelayed"].map(
    lambda x: 1 if x > 15 else 0
)

flightsWithDelays[["DepDelay", "IsDelayed"]].head(10)

In [ ]:
# Cálculo del porcentaje de vuelos retrasados
print(
    "Porcentaje de vuelos retrasados: {:4.2f} %".format(
        100 * flightsWithDelays.IsDelayed.sum() / flightsWithDelays.DepDelay.count()
    )
)

In [ ]:
# Cantidad de vuelos retrasados por transportador
import os
import matplotlib.pyplot as plt  #  type: ignore

if not os.path.exists("../files/images"):
    os.makedirs("../files/images")

flights["IsDelayed"] = flights["DepDelay"].copy()
flights["IsDelayed"] = flights["IsDelayed"].map(lambda x: 0 if pd.isna(x) else x)
flights["IsDelayed"] = flights["IsDelayed"].map(lambda x: int(x > 15))
(flights.groupby("UniqueCarrier").sum())["IsDelayed"].plot.bar(
    color="tab:blue",
    alpha=0.7,
)

plt.gca().spines["left"].set_color("lightgray")
plt.gca().spines["bottom"].set_color("gray")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.savefig(
    "../files/images/delays_by_carrier.png",
)
plt.show()

In [ ]:
# Retrasos por día de la semana
(flights.groupby("DayOfWeek").sum())["IsDelayed"].plot.bar(
    color="tab:blue",
    alpha=0.7,
)

plt.gca().spines["left"].set_color("lightgray")
plt.gca().spines["bottom"].set_color("gray")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.savefig(
    "../files/images/delays_by_day_of_week.png",
)
plt.show()

In [ ]:
# Retrasos por hora del día


flights["hour"] = flights["DepTime"].copy()

flights["hour"] = flights["hour"].map(lambda x: int(x / 100) if not pd.isna(x) else x)

(flights.groupby("hour").sum())["IsDelayed"].plot.bar(
    color="tab:red",
    alpha=0.7,
    figsize=(10, 4),
)

plt.gca().spines["left"].set_color("lightgray")
plt.gca().spines["bottom"].set_color("gray")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.savefig(
    "../files/images/delays_by_hour_of_day.png",
)
plt.show()